# Data Analytics: Tomar decisiones de negocio basadas en datos
**Objetivo:** En este notebook exploraremos cómo formular y verificar hipótesis de negocio, realizar pruebas A/B, y priorizar experimentos para optimizar decisiones basadas en datos.
**Estructura del notebook:**
1. Fundamentos teóricos
2. Ejemplo de prueba A/B con explicación detallada
3. Priorizar hipótesis usando WSJF, ICE y RICE
4. Teoría complementaria sobre pruebas y diseño experimental


## 1. Teórico: Bases de verificación de hipótesis de un negocio y escoger un método experimental
### 1.1 Bases de verificación de hipótesis de un negocio
En el contexto de negocio, una **hipótesis** es una suposición sobre el comportamiento del mercado o de los usuarios que podemos contrastar con datos. Para verificarla:
- **Hipótesis nula (H₀):** Plantea que no hay efecto o diferencia significativa (por ejemplo, **la nueva versión no mejora la tasa de conversión**).
- **Hipótesis alternativa (H₁):** Plantea el efecto o diferencia que esperamos detectar (por ejemplo, **sí hay mejora en la conversión**).
- **Nivel de significancia (α):** Probabilidad de rechazar H₀ cuando es verdadera (error tipo I).
- **Potencia del test (1 - β):** Probabilidad de detectar un efecto real (1 menos la probabilidad de error tipo II).

### 1.2 Escoger un método experimental
Para contrastar hipótesis, podemos elegir distintos diseños:
- **Prueba A/B:** Comparar dos versiones (control vs variante) asignando usuarios aleatoriamente.
- **Prueba A/A:** Validar la estabilidad de la plataforma comparando dos grupos idénticos.
- **Prueba A/B/n:** Comparar múltiples variantes simultáneamente.
- **Diseño multivariante:** Probar combinaciones de cambios en paralelo.
A la hora de escoger, consideramos: recursos disponibles, complejidad, tiempo y riesgos.


### Generación de dataset sintético
Antes de los ejercicios prácticos, generaremos datos simulados que imiten:
- Un experimento A/B con grupos A y B, visitas y órdenes.
- Tablas de priorización de hipótesis (WSJF, ICE, RICE).
Esto nos permitirá ejecutar el código sin depender de archivos externos.


In [1]:
import pandas as pd
import numpy as np

# Semilla para reproducibilidad\ nnp.random.seed(42)

# Dataset para prueba A/B
data_ab = pd.DataFrame({
    'date': pd.date_range(start='2025-01-01', periods=100),
    'group': np.random.choice(['A', 'B'], size=100),
    'visitors': np.random.poisson(100, size=100),
    'orders': np.random.binomial(n=np.random.poisson(100, size=100), p=0.1)
})

# Dataset para WSJF
hypothesis_wsjf = pd.DataFrame({
    'hypothesis': ['Hip1', 'Hip2', 'Hip3', 'Hip4'],
    'user_business_value': [8, 5, 3, 9],
    'time_criticality': [7, 6, 4, 8],
    'risk_reduction_opportunity_enablement': [5, 7, 6, 4],
    'job_size': [3, 2, 5, 4]
})

# Dataset para ICE
hypothesis_ice = pd.DataFrame({
    'hypothesis': ['Hip1', 'Hip2', 'Hip3', 'Hip4'],
    'impact': [9, 6, 4, 8],
    'confidence': [8, 9, 5, 7],
    'efforts': [4, 3, 5, 6]
})

# Dataset para RICE
hypothesis_rice = pd.DataFrame({
    'hypothesis': ['Hip1', 'Hip2', 'Hip3', 'Hip4'],
    'reach': [1000, 800, 600, 1200],
    'impact': [9, 6, 4, 8],
    'confidence': [8, 9, 5, 7],
    'efforts': [4, 3, 5, 6]
})


## 2. Ejemplo de prueba A/B
**Objetivo:** Medir la diferencia en la tasa de conversión entre el grupo **A** (control) y el grupo **B** (variante).
**Pasos:**
1. Resumir datos por grupo.
2. Calcular la métrica clave: órdenes por visitante.
3. Interpretar si hay diferencias relevantes.


In [3]:
# Paso 1: Mostrar primeras filas del dataset
data_ab.head(5)

,date,group,visitors,orders
0,2025-01-01,A,93,14
1,2025-01-02,B,101,8
2,2025-01-03,A,98,11
3,2025-01-04,B,91,9
4,2025-01-05,B,88,9


In [8]:
# Paso 2: Agrupar y sumar visitantes y órdenes por grupo
table_ab = (
    data_ab.drop(columns=['date'])
           .groupby('group', as_index=False)
           .agg({'visitors': 'sum', 'orders': 'sum'})
)

table_ab.head()

,group,visitors,orders
0,A,4875,467
1,B,5156,475


In [9]:
# Paso 3: Calcular tasa de conversión y mostrar resultados
table_ab['orders_to_visitors_ratio'] = table_ab['orders'] / table_ab['visitors']
print(table_ab)

  group  visitors  orders  orders_to_visitors_ratio
0     A      4875     467                  0.095795
1     B      5156     475                  0.092126


## 3. Cómo y por qué priorizamos las hipótesis
Cuando los recursos son limitados, debemos seleccionar las hipótesis que ofrezcan mayor valor esperado al menor costo.
Veremos tres métodos:
- WSJF: pondera valor de negocio y urgencia contra tamaño del trabajo.
- ICE: combina impacto, confianza y facilidad de ejecución.
- RICE: amplía ICE incluyendo alcance (número de usuarios afectados).


### 3.1 WSJF (Weighted Shortest Job First)
Fórmula: \(WSJF = \frac{ValorNegocio + Urgencia + Riesgo}{TamañoTrabajo}\)
Donde:
- **ValorNegocio:** Beneficio esperado.
- **Urgencia:** Críticas del mercado.
- **Riesgo:** Oportunidad de reducción de riesgo.
- **TamañoTrabajo:** Esfuerzo estimado.


In [4]:
hypothesis_wsjf.head()

,hypothesis,user_business_value,time_criticality,risk_reduction_opportunity_enablement,job_size
0,Hip1,8,7,5,3
1,Hip2,5,6,7,2
2,Hip3,3,4,6,5
3,Hip4,9,8,4,4


In [5]:
# Cálculo de WSJF y ordenamiento
hypothesis_wsjf['WSJF'] = (
    hypothesis_wsjf['user_business_value'] + 
    hypothesis_wsjf['time_criticality'] + 
    hypothesis_wsjf['risk_reduction_opportunity_enablement']
) / hypothesis_wsjf['job_size']
print(hypothesis_wsjf[['hypothesis', 'WSJF']].sort_values(by='WSJF', ascending=False))


  hypothesis      WSJF
1       Hip2  9.000000
0       Hip1  6.666667
3       Hip4  5.250000
2       Hip3  2.600000


### 3.2 ICE (Impact, Confidence, Ease)
Fórmula: \(ICE = \frac{Impact \times Confianza}{Esfuerzo}\)
Donde:
- **Impacto:** Crecimiento esperado.
- **Confianza:** Nivel de certeza en la estimación.
- **Esfuerzo:** Recursos necesarios.


In [6]:
hypothesis_ice.head()

,hypothesis,impact,confidence,efforts
0,Hip1,9,8,4
1,Hip2,6,9,3
2,Hip3,4,5,5
3,Hip4,8,7,6


In [7]:
# Cálculo de ICE y ordenamiento
hypothesis_ice['ICE'] = (
    hypothesis_ice['impact'] * hypothesis_ice['confidence']
) / hypothesis_ice['efforts']
print(hypothesis_ice[['hypothesis', 'ICE']].sort_values(by='ICE', ascending=False))


  hypothesis        ICE
0       Hip1  18.000000
1       Hip2  18.000000
3       Hip4   9.333333
2       Hip3   4.000000


### 3.3 RICE (Reach, Impact, Confidence, Effort)
Fórmula: \(RICE = \frac{Reach \times Impact \times Confianza}{Esfuerzo}\)
Donde:
- **Reach:** Usuarios afectados.
- **Impact:** Mejora por usuario.
- **Confianza:** Precisión de las estimaciones.
- **Esfuerzo:** Recursos necesarios.


In [8]:
hypothesis_rice.head()

,hypothesis,reach,impact,confidence,efforts
0,Hip1,1000,9,8,4
1,Hip2,800,6,9,3
2,Hip3,600,4,5,5
3,Hip4,1200,8,7,6


In [9]:
# Cálculo de RICE y ordenamiento
hypothesis_rice['RICE'] = (
    hypothesis_rice['reach'] * hypothesis_rice['impact'] * hypothesis_rice['confidence']
) / hypothesis_rice['efforts']
print(hypothesis_rice[['hypothesis', 'RICE']].sort_values(by='RICE', ascending=False))


  hypothesis     RICE
0       Hip1  18000.0
1       Hip2  14400.0
3       Hip4  11200.0
2       Hip3   2400.0


## 4. Teórico adicional
En esta sección revisamos conceptos clave para diseño y análisis de pruebas:
- **Pruebas A/A:** Validar la consistencia de la plataforma comparando dos grupos idénticos.
- **Errores de tipo I (falsos positivos):** Rechazar H₀ cuando es verdadera, nivel α.
- **Errores de tipo II (falsos negativos):** No rechazar H₀ cuando H₁ es verdadera, nivel β.
- **Comparaciones múltiples (A/B/n):** Ajustes para controlar tasa de error cuando se prueban varias variantes.
- **Cálculo de tamaño de muestra y duración:** Determina cuántos usuarios y tiempo se necesitan para alcanzar potencia deseada.
- **Análisis gráfico de métricas:** Uso de gráficos de líneas, barras y cohortes para visualizar evolución.
- **Definición de benchmarks de industria y audiencia:** Identificar métricas clave por sector y segmentación de usuarios.
